In [6]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from lifelines import CoxPHFitter
import warnings
warnings.filterwarnings("ignore")

print("All packages loaded successfully.")

All packages loaded successfully.


# Preparing Data

In [7]:
# load in datasets
expr = pd.read_csv('../datasets/csv_files/penalized_cox_sig_genes_train.csv')
clinical = pd.read_csv('../datasets/csv_files/clinical_metadata_train.csv')
univariate_results = pd.read_csv('../datasets/csv_files/univariate_cox_sig_train.csv')
expr.head()

,sample_name,KLHL5,BLNK,SOX4,UBE2S,S100P,DEFB132,COLEC12,MICB,EIF4E3,...,FBXL16,CFB,SOD2,TRIM45,MTHFD1,MDK,CEBPD,STAT5A,ACO1,TF
0,GSM1045191,6.072575,5.236619,6.020647,4.445544,5.073908,8.357489,7.386655,5.413113,6.647043,...,4.139581,5.875372,9.288465,4.930487,9.063218,5.188450,8.235638,7.981321,6.697765,6.901910
1,GSM1045192,4.181562,5.595045,6.048561,4.422642,4.521399,3.296474,6.404261,6.659727,4.446082,...,4.614759,5.743448,8.606902,4.745991,6.632947,6.270123,6.229825,7.601242,5.872591,4.818204
2,GSM1045193,5.337126,8.674915,4.799969,4.878597,4.638771,2.953570,4.816429,6.921534,4.109932,...,4.833464,5.258735,4.838683,5.288228,5.941661,5.771265,4.134739,7.501134,4.704279,4.746406
3,GSM1045194,4.675391,4.519967,7.113616,4.498002,5.977879,3.292748,4.931116,5.217874,4.598521,...,4.935647,5.351327,9.178988,5.912944,6.383969,6.423919,7.220704,7.285756,4.650540,4.738504
4,GSM1045195,6.000885,4.955369,5.754098,4.847608,4.839788,6.843711,8.182421,5.132874,6.837103,...,3.937953,5.574557,10.132265,4.677008,9.651571,5.001135,7.528966,8.350870,8.647494,7.463235


In [8]:
# get just gene columns (everything except sample_name)
gene_cols = [col for col in expr.columns if col != 'sample_name']
gene_cols

['KLHL5',
 'BLNK',
 'SOX4',
 'UBE2S',
 'S100P',
 'DEFB132',
 'COLEC12',
 'MICB',
 'EIF4E3',
 'GNS',
 'ARF1',
 'FBXL16',
 'CFB',
 'SOD2',
 'TRIM45',
 'MTHFD1',
 'MDK',
 'CEBPD',
 'STAT5A',
 'ACO1',
 'TF']

In [9]:
# prepare train data
clinical_columns = ['sample_name', 'relapse_free_event', 'relapse_free_time']

train_data = pd.merge(
    clinical[clinical_columns],
    expr[:],
    on='sample_name'
)

train_data = train_data.dropna()
train_data[['relapse_free_event', 'relapse_free_time']] = train_data[['relapse_free_event', 'relapse_free_time']].astype(int)

train_data.head()

,sample_name,relapse_free_event,relapse_free_time,KLHL5,BLNK,SOX4,UBE2S,S100P,DEFB132,COLEC12,...,FBXL16,CFB,SOD2,TRIM45,MTHFD1,MDK,CEBPD,STAT5A,ACO1,TF
17,GSM1045208,0,3026,4.727495,8.036585,7.735311,4.844556,10.187309,3.001555,7.359875,...,5.220802,7.205254,6.141279,6.428617,7.813484,6.349483,5.476177,7.169852,4.799942,4.314603
18,GSM1045209,1,755,4.202276,6.248417,7.967062,6.627002,6.677150,3.119509,5.794930,...,5.929740,7.136334,5.582510,6.394637,7.254114,7.848696,6.687999,6.309295,4.971609,4.921310
19,GSM1045210,0,3014,4.771116,7.606303,8.076291,4.907463,10.623973,2.982960,7.039614,...,5.401792,7.301934,6.262349,6.074443,7.482227,7.003300,6.107935,6.531898,4.555494,4.437907
20,GSM1045211,1,406,3.711394,5.238047,7.511513,5.012260,11.189967,2.820621,6.681315,...,5.254027,5.207979,5.432534,5.916733,6.905146,6.749071,5.820589,6.227280,4.668031,4.252302
21,GSM1045212,0,2225,4.633546,6.438748,7.601258,7.380517,10.322314,2.851260,6.241178,...,4.999650,6.896423,6.716884,6.681431,8.217102,8.426718,6.264166,7.649678,5.458046,4.873803


# Mutivariate Cox Regression

In [10]:
def run_multivariate_cox(df, gene_col, time_col="relapse_free_time", event_col="relapse_free_event"):
    """
    Run one Cox model for all genes and collect results.

    Returns a DataFrame with: gene, beta, HR, 95% CI, p-value
    Sorted by p-value (most significant first).
    """
    results = []
    try:
        cph = CoxPHFitter()
        cph.fit(
            df,
            duration_col=time_col,
            event_col=event_col,
            formula=gene_col
        )

        s = cph.summary

        for gene in s.index:
            results.append({
                "gene":       gene,
                "beta":        round(float(s.loc[gene, "coef"]), 4),           # log(HR)
                "HR":          round(float(s.loc[gene, "exp(coef)"]), 4),      # Hazard Ratio
                "HR_lower_95": round(float(s.loc[gene, "exp(coef) lower 95%"]), 4),
                "HR_upper_95": round(float(s.loc[gene, "exp(coef) upper 95%"]), 4),
                "p_value":     round(float(s.loc[gene, "p"]), 4),
            })

    except Exception as e:
        print(f"  Failed: {e}")

    results_df = pd.DataFrame(results).sort_values("p_value").reset_index(drop=True)
    return results_df

In [11]:
print("Running multivariate Cox regression...\n")
all_results = run_multivariate_cox(train_data, gene_cols)

print("\nAll results (sorted by p-value):")
all_results

Running multivariate Cox regression...


All results (sorted by p-value):


,gene,beta,HR,HR_lower_95,HR_upper_95,p_value
0,S100P,0.2351,1.2651,1.0712,1.4941,0.0056
1,FBXL16,-0.8162,0.4421,0.2363,0.8274,0.0107
2,MICB,-0.4297,0.6507,0.4468,0.9477,0.0251
3,KLHL5,-0.9658,0.3807,0.1486,0.9750,0.0442
4,MTHFD1,-0.6197,0.5381,0.2718,1.0653,0.0753
5,CEBPD,0.5431,1.7213,0.9133,3.2439,0.0930
6,DEFB132,-1.3251,0.2658,0.0474,1.4914,0.1321
7,SOX4,0.6149,1.8495,0.8097,4.2247,0.1445
8,UBE2S,0.2796,1.3226,0.8932,1.9584,0.1627
9,STAT5A,-0.3587,0.6986,0.3568,1.3679,0.2954


In [12]:
significant_genes = all_results[all_results["p_value"] < 0.05].copy()

print(f"{'='*55}")
print(f"Total genes tested:          {len(all_results)}")
print(f"Significant genes (p < 0.05): {len(significant_genes)}")
print(f"{'='*55}\n")
print(significant_genes.to_string(index=False))

Total genes tested:          21
Significant genes (p < 0.05): 4

  gene    beta     HR  HR_lower_95  HR_upper_95  p_value
 S100P  0.2351 1.2651       1.0712       1.4941   0.0056
FBXL16 -0.8162 0.4421       0.2363       0.8274   0.0107
  MICB -0.4297 0.6507       0.4468       0.9477   0.0251
 KLHL5 -0.9658 0.3807       0.1486       0.9750   0.0442


# Filter for Significant Genes

In [13]:
# Paper Section 2.3:
#   HR > 1 → danger gene    (higher expression = higher relapse risk)
#   HR < 1 → protective gene (higher expression = lower relapse risk)

if len(significant_genes) > 0:
    significant_genes["role"] = significant_genes["HR"].apply(
        lambda hr: "danger" if hr > 1 else "protective"
    )
    danger_genes = significant_genes[significant_genes["role"] == "danger"]["gene"].tolist()
    protective_genes = significant_genes[significant_genes["role"] == "protective"]["gene"].tolist()

    print(f"Danger genes     (HR > 1): {danger_genes}")
    print(f"Protective genes (HR < 1): {protective_genes}")
    print(f"\nExpected from paper: danger = TSLP, BIRC5, S100B, MDK, S100P")
    print(f"                     protective = RARRES3, BLNK, ACO1")
else:
    print("No significant genes found — check your data or p-value threshold.")

Danger genes     (HR > 1): ['S100P']
Protective genes (HR < 1): ['FBXL16', 'MICB', 'KLHL5']

Expected from paper: danger = TSLP, BIRC5, S100B, MDK, S100P
                     protective = RARRES3, BLNK, ACO1


# Compute Risk Score

In [18]:
def compute_risk_scores(df, univariate_results, sig_genes, time_col="RFS.time", event_col="RFS"):
    """
    Compute risk score for each patient using univariate Cox betas.
    
    Risk Score = sum(beta_i * expression_i)
    where beta_i comes from the univariate Cox model for gene_i.
    """
    # Extract betas for sig genes only, indexed by gene name
    beta_series = (
        univariate_results
        .loc[univariate_results["gene"].isin(sig_genes)]
        .set_index("gene")["beta"]
    )
    
    # Align: only genes present in both the results and the dataframe
    common_genes = [g for g in sig_genes if g in beta_series.index and g in df.columns]
    if len(common_genes) < len(sig_genes):
        missing = set(sig_genes) - set(common_genes)
        print(f"  Warning: {len(missing)} genes dropped (not in data or results): {missing}")
    
    beta_series = beta_series[common_genes]   # same order

    # Dot product: pandas aligns on index/column names automatically
    risk_scores = df[common_genes] @ beta_series  # shape: (n_patients,)

    # Return a clean DataFrame with scores + survival columns for downstream use
    out = df[[time_col, event_col]].copy()
    out["risk_score"] = risk_scores

    print(f"  Risk scores computed for {len(out)} patients using {len(common_genes)} genes")
    print(f"  Score range: {risk_scores.min():.4f} to {risk_scores.max():.4f}")
    print(f"  Score mean:  {risk_scores.mean():.4f}  std: {risk_scores.std():.4f}")

    return out

In [20]:
sig_genes = all_results.loc[all_results["p_value"] < 0.05, "gene"].tolist()
print(f"{len(sig_genes)} significant genes used for risk scoring\n")

train_scores = compute_risk_scores(
    train_data, 
    univariate_results=all_results,
    sig_genes=sig_genes,
    time_col="relapse_free_time",
    event_col="relapse_free_event",
)

print("\nTrain risk score distribution:")
print(train_scores["risk_score"].describe())

4 significant genes used for risk scoring

  Risk scores computed for 104 patients using 4 genes
  Score range: -12.6037 to -7.3615
  Score mean:  -9.8914  std: 1.1228

Train risk score distribution:
count    104.000000
mean      -9.891434
std        1.122788
min      -12.603653
25%      -10.631709
50%       -9.794339
75%       -9.192517
max       -7.361468
Name: risk_score, dtype: float64


In [21]:
# ============================================================================
# Save results to CSV
# ============================================================================
all_results.to_csv("../datasets/csv_files/multivariate_cox_all_genes_train.csv", index=False)

if len(significant_genes) > 0:
    significant_genes.to_csv("../datasets/csv_files/multivariate_cox_sig_train.csv", index=False)

print("Saved files:")
print("  multivariate_cox_all_genes.csv     — full results for all genes")
print("  multivariate_cox_significant.csv   — significant genes only (p < 0.05)")

Saved files:
  multivariate_cox_all_genes.csv     — full results for all genes
  multivariate_cox_significant.csv   — significant genes only (p < 0.05)
